# Mixture-of-Experts (MoE) with Dynamic Neural Network Integration

This notebook demonstrates a Mixture-of-Experts model that extends the DynamicNetwork framework.

## Features
- **4-Phase Training**: Exploration, Estimation, Main Training, Fine-tuning
- **Expert Efficiency Tracking**: Like DynamicNetwork node efficiency
- **Load Balancing**: Auxiliary loss for balanced expert utilization
- **Health Monitoring**: Cancer/Alzheimer scores for expert health
- **Reward/Penalty System**: Adaptive learning rate based on performance

## Dataset
We use a subset of SlimPajama (or synthetic data as fallback) for next-token prediction.

## 1. Setup

In [ ]:
import sys
import os
import numpy as np

# Add src to path
sys.path.insert(0, os.path.abspath('../src'))

# Set random seed for reproducibility
np.random.seed(42)

print("NumPy version:", np.__version__)

In [ ]:
# Import MoE components
from moe_config import DynamicMoEConfig, ExpertConfig, GatingConfig
from dynamic_moe import DynamicMoE
from tokenizer import SimpleTokenizer
from data_loader import SlimPajamaLoader
from trainer import MoETrainer
from evaluator import MoEEvaluator

print("All modules imported successfully!")

## 2. Configuration

Configure the MoE model with 4 experts and top-2 routing.

In [ ]:
# Create configuration
config = DynamicMoEConfig(
    # Model dimensions
    vocab_size=5000,  # Will be updated after vocabulary building
    embed_dim=128,
    max_seq_len=64,
    seed=42,
    
    # Expert configuration
    expert=ExpertConfig(
        hidden_dim=256,
        num_layers=2,
        activation='relu',
        dropout=0.1,
    ),
    
    # Gating configuration
    gating=GatingConfig(
        num_experts=4,
        top_k=2,
        noise_std=0.1,
        temperature=1.0,
    ),
)

print("Configuration:")
print(f"  - Experts: {config.gating.num_experts}")
print(f"  - Top-k: {config.gating.top_k}")
print(f"  - Embed dim: {config.embed_dim}")
print(f"  - Expert hidden: {config.expert.hidden_dim}")

## 3. Data Loading

Load training data (synthetic or SlimPajama subset).

In [ ]:
# Initialize tokenizer
tokenizer = SimpleTokenizer(vocab_size=5000)

# Initialize data loader
data_loader = SlimPajamaLoader(
    subset_size=3000,  # Use 3000 samples for demo
    max_seq_len=64,
    tokenizer=tokenizer,
    seed=42
)

# Load data (will use synthetic if HuggingFace not available)
X, y = data_loader.load_dataset(build_vocab=True)

print(f"\nData loaded:")
print(f"  - Samples: {len(X)}")
print(f"  - Input shape: {X.shape}")
print(f"  - Target shape: {y.shape}")
print(f"  - Vocabulary size: {tokenizer.actual_vocab_size}")

In [ ]:
# Split into train/validation
X_train, y_train, X_val, y_val = data_loader.train_val_split(X, y, val_ratio=0.1)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

In [ ]:
# Show sample data
print("Sample input (first 20 tokens):")
print(X_train[0][:20])

print("\nDecoded sample:")
print(tokenizer.decode(X_train[0]))

## 4. Model Creation

Create the Dynamic MoE model.

In [ ]:
# Update config with actual vocab size
config.vocab_size = tokenizer.actual_vocab_size

# Create model
model = DynamicMoE(config)

print("Model created!")
print(f"  - Vocabulary: {model.vocab_size}")
print(f"  - Embedding dim: {model.embed_dim}")
print(f"  - Number of experts: {len(model.moe_layer.experts)}")

## 5. Training

Train the model using 4-phase training:
1. **Exploration**: High routing noise to discover expert specializations
2. **Estimation**: Estimate epochs needed based on efficiency convergence
3. **Main Training**: Adaptive learning rate with reward/penalty
4. **Fine-tuning**: Frozen routing, polish expert weights

In [ ]:
# Train the model
result = model.fit(
    X_train, y_train,
    X_val, y_val,
    verbose=True
)

In [ ]:
# Training summary
print("\nTraining Summary:")
print(f"  - Epochs completed: {result.epochs_completed}")
print(f"  - Final loss: {result.final_cost:.4f}")
print(f"  - Final perplexity: {result.final_perplexity:.2f}")
print(f"  - Best loss: {result.best_cost:.4f}")
print(f"  - Rewards/Penalties: {result.total_rewards}/{result.total_penalties}")

## 6. Training Visualization

In [ ]:
try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Loss curve
    ax = axes[0, 0]
    ax.plot(result.cost_history)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training Loss')
    ax.grid(True)
    
    # Perplexity
    ax = axes[0, 1]
    ax.plot(result.perplexity_history)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Perplexity')
    ax.set_title('Perplexity')
    ax.grid(True)
    
    # Health scores
    ax = axes[1, 0]
    ax.plot(result.cancer_score_history, label='Cancer (Domination)')
    ax.plot(result.alzheimer_score_history, label='Alzheimer (Death)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Score')
    ax.set_title('Expert Health Scores')
    ax.legend()
    ax.grid(True)
    
    # Efficiency
    ax = axes[1, 1]
    ax.plot(result.efficiency_history)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Efficiency')
    ax.set_title('Mean Expert Efficiency')
    ax.grid(True)
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("matplotlib not available for plotting")
    print("\nText summary:")
    print(f"  - Loss range: {min(result.cost_history):.4f} - {max(result.cost_history):.4f}")
    print(f"  - Final efficiency: {result.efficiency_history[-1]:.3f}")
    print(f"  - Final cancer score: {result.cancer_score_history[-1]:.3f}")
    print(f"  - Final alzheimer score: {result.alzheimer_score_history[-1]:.3f}")

## 7. Expert Analysis

Analyze expert utilization and specialization.

In [ ]:
# Get expert statistics
stats = model.get_expert_stats()

print("Expert Load Distribution:")
for i, load in enumerate(stats['gating']['expert_loads']):
    print(f"  Expert {i}: {load:.2%}")

print(f"\nLoad Balance Loss: {stats['gating']['load_balance_loss']:.4f}")
print(f"Load Entropy: {stats['gating']['load_entropy']:.3f}")

In [ ]:
# Expert efficiencies
print("Expert Efficiencies:")
for exp in stats['experts']:
    print(f"  Expert {exp['expert_id']}: {exp['efficiency']:.3f}")
    print(f"    - Utilization: {exp['utilization']:.2%}")
    print(f"    - Gradient magnitude: {exp['gradient_magnitude']:.4f}")
    print(f"    - Dead nodes: {exp['dead_nodes']}")

In [ ]:
# Visualize expert loads
try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    loads = stats['gating']['expert_loads']
    efficiencies = [e['efficiency'] for e in stats['experts']]
    x = range(len(loads))
    
    # Load distribution
    ax = axes[0]
    bars = ax.bar(x, loads, color='steelblue')
    ax.axhline(y=1/len(loads), color='r', linestyle='--', label='Uniform')
    ax.set_xlabel('Expert')
    ax.set_ylabel('Load')
    ax.set_title('Expert Load Distribution')
    ax.set_xticks(x)
    ax.legend()
    
    # Efficiency
    ax = axes[1]
    bars = ax.bar(x, efficiencies, color='forestgreen')
    ax.axhline(y=0.5, color='r', linestyle='--', label='Threshold')
    ax.set_xlabel('Expert')
    ax.set_ylabel('Efficiency')
    ax.set_title('Expert Efficiency')
    ax.set_xticks(x)
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("matplotlib not available")

## 8. Health Monitoring

Check the health status of the MoE model.

In [ ]:
# Get health report
health = model.get_health_report()

print("MoE Health Report")
print("=" * 40)
print(f"State: {health.state.upper()}")
print(f"Overall Health: {health.overall_health:.2%}")
print(f"\nHealth Scores:")
print(f"  - Cancer (Domination): {health.cancer_score:.3f}")
print(f"  - Alzheimer (Death): {health.alzheimer_score:.3f}")
print(f"\nDiagnosis: {health.diagnosis}")

if health.recommendations:
    print(f"\nRecommendations:")
    for rec in health.recommendations:
        print(f"  - {rec}")

## 9. Evaluation

In [ ]:
# Create evaluator
evaluator = MoEEvaluator(model)

# Evaluate on validation set
metrics = evaluator.evaluate(X_val, y_val)

print("Validation Metrics:")
print(f"  - Loss: {metrics['loss']:.4f}")
print(f"  - Perplexity: {metrics['perplexity']:.2f}")
print(f"  - Accuracy: {metrics['accuracy']:.2%}")

In [ ]:
# Compare with single expert baseline
comparison = evaluator.compare_with_baseline(X_val, y_val, baseline_type='single_expert')

print("\nComparison with Single Expert Baseline:")
print(f"  MoE Loss: {comparison['moe']['loss']:.4f}")
print(f"  Baseline Loss: {comparison['baseline']['loss']:.4f}")
print(f"  Improvement: {comparison['improvements']['loss']:.1%}")
print(f"\n  MoE Perplexity: {comparison['moe']['perplexity']:.2f}")
print(f"  Baseline Perplexity: {comparison['baseline']['perplexity']:.2f}")
print(f"  Improvement: {comparison['improvements']['perplexity']:.1%}")

## 10. Text Generation

In [ ]:
# Generate text from prompts
prompts = [
    "the cat",
    "a big",
    "once upon",
]

print("Text Generation Examples:")
print("=" * 40)

for prompt in prompts:
    # Encode prompt
    prompt_ids = tokenizer.encode(prompt, max_length=10, add_special_tokens=True, padding=False)
    prompt_ids = prompt_ids.reshape(1, -1)
    
    # Generate
    generated_ids = model.generate(
        prompt_ids,
        max_new_tokens=20,
        temperature=0.8,
        top_k=50
    )
    
    # Decode
    generated_text = tokenizer.decode(generated_ids[0])
    
    print(f"\nPrompt: '{prompt}'")
    print(f"Generated: '{generated_text}'")

## 11. Full Evaluation Report

In [ ]:
# Generate comprehensive report
report = evaluator.generate_report(X_val, y_val)
print(report)

## 12. Conclusions

### Key Observations

1. **4-Phase Training**: The model successfully went through all four training phases:
   - Exploration discovered expert specializations
   - Estimation predicted training epochs
   - Main training used adaptive learning rate
   - Fine-tuning polished the weights

2. **Load Balancing**: The auxiliary loss helped maintain balanced expert utilization

3. **Health Monitoring**: Cancer/Alzheimer scores tracked expert health throughout training

4. **Performance**: MoE outperformed the single-expert baseline (see comparison metrics)

### Next Steps

- Try with larger datasets (full SlimPajama subset)
- Experiment with more experts
- Increase model capacity (embed_dim, hidden_dim)
- Add attention mechanisms for better context modeling

In [ ]:
print("Notebook complete!")